In [1]:
import sys
sys.path.insert(0, '../')
import torch.nn as nn
import torch
import pandas as pd
import numpy as np
import os
from src.load_dataset import load_dataset, select_normalizer
from src.config import models_features_per

from src.utils import per_error

In [2]:
class VoltNeural(nn.Module):
    def __init__(self,
                input_dim,
                hidden_dim,
                output,
                device):
        
        super(VoltNeural, self).__init__()
        self.input_layer    = nn.Linear(input_dim,   hidden_dim,  device=device)
        self.hidden_layer   = nn.Linear(hidden_dim,  hidden_dim,  device=device)
        self.hidden_layer1  = nn.Linear(hidden_dim,  hidden_dim,  device=device)
        self.output_layer   = nn.Linear(hidden_dim, 1, device=device)
    
        self.relu          = nn.ReLU()
        self.sigmoid       = nn.Sigmoid()

    def forward(self, x):
        x = self.input_layer(x)
        x = self.relu(x)
        x = self.hidden_layer(x)
        x = self.relu(x)
        x = self.hidden_layer1(x)
        x = self.relu(x)
        x = self.output_layer(x)
        x = self.relu(x)

        return x

In [3]:
def normalize_create_training_data(train, test, blank_norm=False, remove_outlier=None, normalizer_type='mean_std'):

    # Remove outlier only from the training dataset
    if remove_outlier=='all':
        
        train = train[train['file'].apply(lambda x: False if (x.split('/')[-1].replace('.txt', '') in ouliners_to_remove) else True)]
        test  = test[test['file'].apply(lambda x: False if (x.split('/')[-1].replace('.txt', '') in ouliners_to_remove) else True)]

    elif remove_outlier=='train_only':
        train = train[train['file'].apply(lambda x: False if (x.split('/')[-1].replace('.txt', '') in ouliners_to_remove) else True)]
        
    train = train.reset_index(drop=True)
    test  = test.reset_index(drop=True)
    
    X_train = train.drop(columns=['file']).copy()
    X_test  = test.drop(columns=['file']).copy()

    columns       = X_train.columns
    
    y_train = train['file'].apply(lambda x: int(x.split('_')[-2].replace('cbz','')))
    y_test  = test['file'].apply(lambda x: int(x.split('_')[-2].replace('cbz','')))

    assert (X_train.index.values == y_train.index.values).all()

    if normalizer_type!=None:
        scaler  = select_normalizer(normalizer_type)
    
        if blank_norm: scaler.fit(X_train[y_train==0])
        else: scaler.fit(X_train)
    
        
        X_train = pd.DataFrame(scaler.transform(X_train), columns=columns)
        X_test  = pd.DataFrame(scaler.transform(X_test),  columns=columns)

    else:
        scaler = None

    X_train.rename(columns={"PH": 'univariate, max(S)', 'signal_std':'univariate, std(S)', 'signal_mean':'univariate, mean(S)', 'peak area':'univariate, area(S)', \
                        'dS_dV_area':'univariate, area(dS/dV)', 'dS_dV_max_peak':'univariate, max(dS/dV)', 'dS_dV_min_peak':'univariate, min(dS/dV)',\
                    'dS_dV_peak_diff':'univariate, max(dS/dV) - min(dS/dV)', \
                    'peak V':'univariate, V_max(S)', 'dS_dV_max_V':'univariate, V_max(dS/dV)', 'dS_dV_min_V':'univariate, V_min(dS/dV)',\
        }, inplace = True)

    X_test.rename(columns={"PH": 'univariate, max(S)', 'signal_std':'univariate, std(S)', 'signal_mean':'univariate, mean(S)', 'peak area':'univariate, area(S)', \
                        'dS_dV_area':'univariate, area(dS/dV)', 'dS_dV_max_peak':'univariate, max(dS/dV)', 'dS_dV_min_peak':'univariate, min(dS/dV)',\
                    'dS_dV_peak_diff':'univariate, max(dS/dV) - min(dS/dV)', \
                    'peak V':'univariate, V_max(S)', 'dS_dV_max_V':'univariate, V_max(dS/dV)', 'dS_dV_min_V':'univariate, V_min(dS/dV)',\
        }, inplace = True)

   

    return (X_train, X_test, y_train, y_test), scaler


def load_dataset_train_test_splitted(filename, load_dataset_name=['ML1', 'ML2', 'ML4']):
    dataset = {}

    if 'ML1' in load_dataset_name:
        dataset['ML1'] = pd.read_excel(f'../dataset/ML1_ML2/2024_02_19_ML1/{filename}.xlsx')

    if 'ML2' in load_dataset_name:
         dataset['ML2'] = pd.read_excel(f'../dataset/ML1_ML2/2024_02_22_ML2/{filename}.xlsx')
    
    if 'ML4' in load_dataset_name:
        dataset['ML4'] = pd.read_excel(f'../dataset/ML4/{filename}.xlsx')

    return dataset

In [4]:
os.path.isfile('../dataset/ML1_ML2/2024_02_19_ML1/feature_extraction_vwidth_0.15_root_min_training_noisy.xlsx')
os.path.isfile('../dataset/ML1_ML2/2024_02_19_ML1/feature_extraction_vwidth_0.15_root_min_training.xlsx')

True

In [5]:
# load dataset
# (ML1_X_train, ML1_X_test, ML1_y_train, ML1_y_test), _  = load_dataset('/Users/sangam/Desktop/Epilepsey/Code/vgramreg/dataset/ML1_ML2/2024_02_19_ML1', standardize_type='mean_std')
# (ML2_X_train, ML2_X_test, ML2_y_train, ML2_y_test), _  = load_dataset('/Users/sangam/Desktop/Epilepsey/Code/vgramreg/dataset/ML1_ML2/2024_02_22_ML2', standardize_type='mean_std')
# (ML4_X_train, ML4_X_test, ML4_y_train, ML4_y_test), _  = load_dataset('/Users/sangam/Desktop/Epilepsey/Code/vgramreg/dataset/ML4', standardize_type='mean_std')
vwidth      = 0.15
all_dataset = True
root_min    = True

ML1_noisy_train, ML2_noisy_train, ML4_noisy_train = list(load_dataset_train_test_splitted(f"feature_extraction_vwidth_{vwidth}_{'root_min_' if root_min else ''}training_noisy").values())
ML1_train, ML2_train, ML4_train = list(load_dataset_train_test_splitted(f"feature_extraction_vwidth_{vwidth}_{'root_min_' if root_min else ''}training").values())
ML1_test, ML2_test, ML4_test    = list(load_dataset_train_test_splitted(f"feature_extraction_vwidth_{vwidth}_{'root_min_' if root_min else ''}testing").values())


In [6]:
ouliners_to_remove = ['2024_02_19_cbz08_40',
                      '2024_02_19_cbz00_15',
                      '2024_02_19_cbz08_37',
                      '2024_02_22_cbz00_31',
                     '2024_02_22_cbz16_21',
                     '2024_02_22_cbz08_10',
                     '2024_02_22_cbz00_01',
                     '2024_02_22_cbz08_01']

In [7]:
data_propery    = 'augmentation' # noisy, augmentation, and noiseless
blank_norm      = False
remove_outlier  = 'train_only'   #None, train_only, all
normalizer_type = 'mean_std'
use_combat      = False
add_alpha       = False

if data_propery=='noisy':
    (ML1_X_train, ML1_X_test, ML1_y_train, ML1_y_test), ML1_scalar = normalize_create_training_data(ML1_noisy_train, ML1_test, blank_norm, remove_outlier, normalizer_type=normalizer_type)
    (ML2_X_train, ML2_X_test, ML2_y_train, ML2_y_test), ML2_scalar = normalize_create_training_data(ML2_noisy_train, ML2_test, blank_norm, remove_outlier, normalizer_type=normalizer_type)
    if all_dataset:
        (ML4_X_train, ML4_X_test, ML4_y_train, ML4_y_test), ML4_scalar = normalize_create_training_data(ML4_noisy_train, ML4_test, blank_norm, remove_outlier, normalizer_type=normalizer_type)

elif data_propery=='augmentation':
    ML1_train_combined = pd.concat([ML1_noisy_train, ML1_train])
    ML2_train_combined = pd.concat([ML2_noisy_train, ML2_train])

    if all_dataset:
        ML4_train_combined = pd.concat([ML4_noisy_train, ML4_train])
    
    (ML1_X_train, ML1_X_test, ML1_y_train, ML1_y_test), ML1_scalar = normalize_create_training_data(ML1_train_combined, ML1_test, blank_norm, remove_outlier, normalizer_type=normalizer_type)
    (ML2_X_train, ML2_X_test, ML2_y_train, ML2_y_test), ML2_scalar = normalize_create_training_data(ML2_train_combined, ML2_test, blank_norm, remove_outlier, normalizer_type=normalizer_type)
    if all_dataset:
        (ML4_X_train, ML4_X_test, ML4_y_train, ML4_y_test), ML4_scalar = normalize_create_training_data(ML4_train_combined, ML4_test, blank_norm, remove_outlier, normalizer_type=normalizer_type)
    
else:
    (ML1_X_train, ML1_X_test, ML1_y_train, ML1_y_test), ML1_scalar = normalize_create_training_data(ML1_train, ML1_test, blank_norm, remove_outlier, normalizer_type=normalizer_type)
    (ML2_X_train, ML2_X_test, ML2_y_train, ML2_y_test), ML2_scalar = normalize_create_training_data(ML2_train, ML2_test, blank_norm, remove_outlier, normalizer_type=normalizer_type)
    if all_dataset:
        (ML4_X_train, ML4_X_test, ML4_y_train, ML4_y_test), ML4_scalar = normalize_create_training_data(ML4_train, ML4_test, blank_norm, remove_outlier, normalizer_type=normalizer_type)
        

In [8]:
X_train = pd.concat([ML1_X_train, ML2_X_train, ML4_X_train], axis=0) if all_dataset else pd.concat([ML1_X_train, ML2_X_train], axis=0)
y_train = pd.concat([ML1_y_train, ML2_y_train, ML4_y_train], axis=0) if all_dataset else pd.concat([ML1_y_train, ML2_y_train], axis=0)
X_test  = pd.concat([ML1_X_test,  ML2_X_test,  ML4_X_test], axis=0) if all_dataset else pd.concat([ML1_X_test,  ML2_X_test], axis=0)
y_test  = pd.concat([ML1_y_test,  ML2_y_test,  ML4_y_test], axis=0) if all_dataset else pd.concat([ML1_y_test,  ML2_y_test], axis=0)

if add_alpha:
    X_train = pd.concat([X_train, MLA_X, MLA_X_noisy], axis=0)
    y_train = pd.concat([y_train, MLA_y, MLA_y_noisy], axis=0)


indx_shuffle = np.random.permutation(range(len(X_train)))
X_train      = X_train.iloc[indx_shuffle]
y_train      = y_train.iloc[indx_shuffle]

In [9]:
X_train = X_train[['univariate, max(S)']].to_numpy()
X_test  = X_test[['univariate, max(S)']].to_numpy()
y_train = y_train.to_numpy()[..., np.newaxis]
y_test  = y_test.to_numpy()[..., np.newaxis]

In [10]:
X_train.shape

(424, 1)

In [11]:
from sklearn.linear_model import LinearRegression, SGDRegressor

In [12]:
# Train the model
iteration    = 100000
lr           = 1e-3
batch_size   = 8
num_datasize = len(X_train)
num_hidden_layer = 4


device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
model  = VoltNeural(1, num_hidden_layer, 1, device)

#Set the weights in the neural network
# model.input_layer.weight.data = torch.tensor([[10.0]], dtype=torch.float32).to(device)
# model.input_layer.bias.data   = torch.tensor([[15.0]], dtype=torch.float32).to(device)


# Define loss function
criteron  = nn.MSELoss()

# Define optimizer
optim     = torch.optim.Adam(model.parameters(),lr=lr)
num_step  = num_datasize // batch_size

for epoch in range(iteration):
    model.train()

    for i in range(num_step):
        start = i*batch_size
        end   = start + batch_size

        X_    = torch.tensor(X_train[start:end], dtype=torch.float32).to(device)
        y_    = torch.tensor(y_train[start:end], dtype=torch.float32).to(device)

        # Train Deep Learning Model
        output = model(X_) 
        loss   = criteron(output, y_)

        optim.zero_grad()
        loss.backward()
        optim.step()

    if (epoch%100 == 0):
        model.eval()
        # y_test_pred = model(torch.tensor(X_test, dtype=torch.float32).to(device))
        # test_loss   = nn.MSELoss()(y_test_pred, torch.tensor(y_test).to(device))
        print(f"Epoch:{epoch} | Loss:{loss.item():.2f} ")
        # model.train()

cuda
Epoch:0 | Loss:110.66 
Epoch:100 | Loss:1.58 
Epoch:200 | Loss:1.28 
Epoch:300 | Loss:1.09 
Epoch:400 | Loss:1.04 
Epoch:500 | Loss:1.02 
Epoch:600 | Loss:1.02 
Epoch:700 | Loss:1.02 
Epoch:800 | Loss:1.01 
Epoch:900 | Loss:1.01 
Epoch:1000 | Loss:1.01 
Epoch:1100 | Loss:1.01 
Epoch:1200 | Loss:1.01 
Epoch:1300 | Loss:1.01 
Epoch:1400 | Loss:1.01 
Epoch:1500 | Loss:1.01 
Epoch:1600 | Loss:1.01 
Epoch:1700 | Loss:1.01 
Epoch:1800 | Loss:1.01 
Epoch:1900 | Loss:1.01 



KeyboardInterrupt



# Calculate Percentage error and R2 Score

In [13]:
from src.load_models import select_model

In [14]:
y_LOD = 0.9117010154341669
model.eval()
y_nn_pred = model(torch.tensor(X_test, dtype=torch.float32).to(device))
per_error(y_test, y_nn_pred.detach().cpu().numpy(), y_LOD)

23.087511389211414

In [15]:
for model_name in ['Linear', 'KNN', 'RF', 'GP', 'DT', 'SVM']:
    model_trad = select_model(model_name)
    model_trad.fit(X_train, y_train)
    y_pred = model_trad.predict(X_test).squeeze()
    print(model_name, per_error(y_test.squeeze(), y_pred, y_LOD))

Linear 60.119991513148975
KNN 12.501082954642515


/nfs/stak/users/buddhacs/hpc-share/anaconda3/envs/vgramre/lib/python3.9/site-packages/sklearn/base.py:1152: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RF 12.9709631828408
GP 16.179751197107034
DT 8.675799086757989
SVM 33.096163446625724


/nfs/stak/users/buddhacs/hpc-share/anaconda3/envs/vgramre/lib/python3.9/site-packages/sklearn/utils/validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [ ]:
nn.MSELoss()(torch.tensor(lin_model.predict(X_test), dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32))

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Sample dataset
x_rand   = np.linspace(-10, 10, 1000)[..., np.newaxis] # np.random.uniform(low=1, high=10, size=(1000,1))#np.random.randn(10000, 1)#
y_output = 15 * x_rand + 10 + np.random.normal(0, 1, size=x_rand.shape)

ind_shuffle = np.random.choice(range(len(x_rand)), len(x_rand))
x_rand      = x_rand[ind_shuffle]
y_output    = y_output[ind_shuffle]


X = torch.tensor(X_train, dtype=torch.float32).to(device)
y = torch.tensor(y_train, dtype=torch.float32).to(device)  # y = 2*x + 1


# Define a simple linear model
model = nn.Linear(1, 1)  # 1 input feature, 1 output
model.to(device)

# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.001)

# Training the model
for epoch in range(10000):
    model.train()
    
    # Forward pass
    y_pred = model(X)

    print(y_pred, y)
    break
    
    # Compute loss
    loss = criterion(y_pred, y)
    
    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Get the learned weights and bias
with torch.no_grad():
    weights = model.weight.data.to('cpu').numpy()
    bias    = model.bias.data.to('cpu').numpy()

print("Learned weights:", weights)
print("Learned bias:", bias)


In [ ]:
lin_sdg_model = SGDRegressor()

In [ ]:
lin_sdg_model.fit(X.to('cpu'), y.to('cpu'))

In [ ]:
lin_sdg_model.coef_, lin_sdg_model.intercept_